# Week 2 – Exercise – Individual – Data Quality Audit
## Name: Anushka Sawant
## Matriculation Number: 100006644

## **TASK 1**
### **Dataset Description & Business use case:**
**Dataset Overview:**
The dataset comprises 11 transaction records — including one confirmed duplicate — spanning nine attributes:
`transaction_id`, `customer_id`, `transaction_date`, `amount`, `currency`, `payment_method`, `status`, `region`, and `last_updated`.
While small in volume, the dataset reflects the structure of a production-grade financial ledger and serves as a representative sample for quality assessment.

**Business Use Case:**
This data is consistent with a Financial Auditing and Revenue Recognition pipeline. In an operational context, it supports tracking of regional sales performance, payment method distribution, and transaction lifecycle status. Downstream consumers of this data likely include finance teams running month-end close reports, BI dashboards monitoring KPIs, and compliance functions validating currency and payment records.

**Impact of Current Data Quality Issues:**
The existing quality gaps — including missing identifiers, invalid amounts, inconsistent region labels, and duplicate records — introduce compounding risks across the business:
- **Tax & Compliance:** Malformed or missing amounts distort taxable revenue calculations and may trigger audit flags.
- **KPI Reliability:** Duplicate transactions inflate sales figures, while null customer IDs break attribution models used in performance reporting.
- **Regional Analysis:** Inconsistent region labels fragment geographic aggregations, making cross-region comparisons unreliable.
- **Pipeline Stability:** Type inconsistencies in `amount` and `transaction_date` will cause downstream failures in any automated ETL or ML feature engineering step.

**Recommendation:**
Before this dataset can be used in any reporting or analytical workflow, a structured remediation pass — covering deduplication, type casting, imputation, and label standardization — is essential to restore data integrity and business trust.

### Importing Packages

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.request import urlretrieve

### Data Loading

In [6]:
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/AnushkaSDBI/Data-Management/main"

required_files = [
    ("Data/week2_customer_transactions_messy.csv", "Data/week2_customer_transactions_messy.csv"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

Downloaded: Data/week2_customer_transactions_messy.csv
Bootstrap complete.


In [7]:
repo_root = Path().resolve()
csv_path = repo_root / "Data" / "week2_customer_transactions_messy.csv"

print("Expected CSV path:", csv_path)
print("File exists:", csv_path.exists())

df = pd.read_csv(csv_path)
display(df)

Expected CSV path: /content/Data/week2_customer_transactions_messy.csv
File exists: True


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
7,T0007,C106,2026-02-30,47.00,EUR,card,completed,DE,2026-02-15
8,T0008,C107,2026-01-10,1000000.00,EUR,card,completed,DE,2026-01-10
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN


## Data Description

In [8]:
# Dataset information
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    11 non-null     object 
 1   customer_id       10 non-null     object 
 2   transaction_date  11 non-null     object 
 3   amount            10 non-null     float64
 4   currency          11 non-null     object 
 5   payment_method    10 non-null     object 
 6   status            11 non-null     object 
 7   region            11 non-null     object 
 8   last_updated      10 non-null     object 
dtypes: float64(1), object(8)
memory usage: 924.0+ bytes


None

In [9]:
# Data Description
display(df.describe())

,amount
count,10.000000
mean,100056.457000
std,316207.939002
min,-35.000000
25%,19.990000
50%,49.550000
75%,112.872500
max,1000000.000000


In [10]:
# Null values
print(df.isnull().sum())

transaction_id      0
customer_id         1
transaction_date    0
amount              1
currency            0
payment_method      1
status              0
region              0
last_updated        1
dtype: int64


## **TASK 2**
### **Issues by Dimension**

In [11]:
quality_report = pd.DataFrame([
    ['Null customer IDs', 'Completeness', 'Breaks customer segmentation and attribution models.'],
    ['Duplicate transactions', 'Uniqueness', 'Inflates revenue metrics and distorts trend analysis.'],
    ['Invalid date entries', 'Validity', 'Corrupts time-series analysis and reporting periods.'],
    ['Inconsistent region labels', 'Consistency', 'Prevents reliable geographic segmentation of data.']
], columns=['Issue', 'Dimension', 'Impact'])
quality_report

,Issue,Dimension,Impact
0,Null customer IDs,Completeness,Breaks customer segmentation and attribution m...
1,Duplicate transactions,Uniqueness,Inflates revenue metrics and distorts trend an...
2,Invalid date entries,Validity,Corrupts time-series analysis and reporting pe...
3,Inconsistent region labels,Consistency,Prevents reliable geographic segmentation of d...


## **TASK 3**
### **KPI Calculations**

In [15]:
# Completeness Rate (Customer ID and Amount)
missing_count = df[['customer_id', 'amount']].isnull().any(axis=1).sum()
completeness_rate = ((len(df) - missing_count) / len(df)) * 100

# Duplication Rate
dup_count = df.duplicated().sum()
duplication_rate = (dup_count / len(df)) * 100

# Validity Error Rate (Amount > 0 and realistic)
valid_amount = df['amount'].apply(lambda x: 0 < x < 1000000 if pd.notnull(x) else False)
error_rate = ((len(df) - valid_amount.sum()) / len(df)) * 100

# # Consistency Rate (Region Labels)
valid_regions = {'DE', 'FR', 'NL'}  # define your expected values
inconsistent_region = df['region'].str.strip().apply(
    lambda x: x not in valid_regions if pd.notnull(x) else False
)
inconsistency_rate = (inconsistent_region.sum() / len(df)) * 100

print(f"KPI Summary:")
print(f"1. Completeness Rate: {completeness_rate:.2f}%")
print(f"2. Duplication Rate: {duplication_rate:.2f}%")
print(f"3. Error Rate (Validity): {error_rate:.2f}%")
print(f"4. Inconsistency Rate (Region): {inconsistency_rate:.2f}%")

KPI Summary:
1. Completeness Rate: 81.82%
2. Duplication Rate: 9.09%
3. Error Rate (Validity): 36.36%
4. Inconsistency Rate (Region): 18.18%


### **KPI Interpretation**
**1. Completeness Rate (81.82%):**

Meaning: Roughly 1 in 5 records is missing a customer_id, an amount, or both.

Impact: This level of missing data is significant. In financial reporting, it creates "revenue leakage" — transactions that can't be traced to a customer or assigned a correct value.

**2. Duplication Rate (9.09%):**

Meaning: Nearly 1 in 10 rows is an exact copy of another row in the dataset.

Impact: This points to a likely problem in the ingestion pipeline — such as a job running more than once or missing deduplication logic — and will cause sales figures to be overstated by close to 10%.

**3. Error Rate / Validity (36.36%):**

Meaning: More than a third of records fail the amount validity check — values are either non-numeric, negative, or unrealistically large (above $1,000,000).

Impact: This is the most critical issue. It signals that the amount column is fundamentally unreliable and may require a full data audit or a rethink of how that field is being captured at the source.

**4. Inconsistency Rate (18.18%):**

Meaning: 18% of records contain region labels that don't conform to the expected set of valid values — likely due to free-text entry, abbreviations, or casing variations (e.g. "DE", "de").

Impact: This undermines any geographic segmentation or regional reporting. Aggregations by region will be fragmented and misleading until a standardized lookup or input validation is enforced upstream.

## **TASK 4**
### **Validation Rules**

In [ ]:
rules={
'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip()==''),
'amount_non_negative': pd.to_numeric(df['amount'], errors='coerce')<0,
'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),
}
pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T

,affected_rows
transaction_id_required,0
amount_non_negative,1
transaction_date_valid,1


## **TASK 5**
### **Audit Summary**

In [16]:
audit_data = [
    ['Invalid/Outlier Amounts', '36.36%', 'Critical', 'Verify values > $1M and handle negative entries.'],
    ['Missing ID or Amount', '18.18%', 'High', 'Isolate rows missing customer_id for reconciliation.'],
    ['Duplicate Transactions', '9.09%', 'Medium', 'Apply drop_duplicates() to stabilize revenue reporting.'],
    ['Inconsistent Region Labels', '18.18%', 'Medium', 'Standardize values against a valid_regions lookup and enforce input validation.']
]

audit_summary = pd.DataFrame(audit_data, columns=['issue_type', 'affected_rows', 'severity', 'recommended_next_action'])
audit_summary

,issue_type,affected_rows,severity,recommended_next_action
0,Invalid/Outlier Amounts,36.36%,Critical,Verify values > $1M and handle negative entries.
1,Missing ID or Amount,18.18%,High,Isolate rows missing customer_id for reconcili...
2,Duplicate Transactions,9.09%,Medium,Apply drop_duplicates() to stabilize revenue r...
3,Inconsistent Region Labels,18.00%,Medium,Standardize values against a valid_regions loo...


## **TASK 6**
### **Recommended Cleaning Actions for next week:**

In [20]:
remediation_plan = pd.DataFrame([
    [1, 'Schema Normalization', 'Critical', 'Standardize all column headers with .str.strip().lower() to prevent KeyErrors caused by whitespace or casing.'],
    [2, 'Record Deduplication', 'High', 'Run drop_duplicates() to eliminate the ~9% redundant rows and enforce unique transaction_ids throughout.'],
    [3, 'Financial Type Casting', 'Critical', 'Cast amount to numeric and transaction_date to datetime using errors="coerce" to surface unparseable entries.'],
    [4, 'Strategic Imputation', 'Medium', 'Replace missing amount values with the per-category median to maintain the underlying distribution.'],
    [5, 'Label Standardization', 'Low', 'Apply a mapping dictionary (e.g., {"electr": "Electronics"}) to consolidate fragmented category labels.']
], columns=['step', 'action_item', 'priority', 'expected_outcome'])

display(remediation_plan)

,step,action_item,priority,expected_outcome
0,1,Schema Normalization,Critical,Standardize all column headers with .str.strip...
1,2,Record Deduplication,High,Run drop_duplicates() to eliminate the ~9% red...
2,3,Financial Type Casting,Critical,Cast amount to numeric and transaction_date to...
3,4,Strategic Imputation,Medium,Replace missing amount values with the per-cat...
4,5,Label Standardization,Low,"Apply a mapping dictionary (e.g., {""electr"": ""..."


## **Bonus (Audit Function)**

In [18]:
rules = {
    'Invalid/Outlier Amounts': ~df['amount'].apply(lambda x: 0 < x < 1000000 if pd.notnull(x) else False),
    'Missing ID or Amount': df[['customer_id', 'amount']].isnull().any(axis=1),
    'Duplicate Transactions': df.duplicated(),
    'Inconsistent Region Labels': df['region'].str.strip().apply(
        lambda x: x not in {'DE', 'FR', 'NL'} if pd.notnull(x) else False
    )
}

def summarize_rule_violations(rule_dictionary):
    return pd.DataFrame({
        'rule_name': list(rule_dictionary.keys()),
        'affected_rows': [int(mask.sum()) for mask in rule_dictionary.values()]
    }).sort_values('affected_rows', ascending=False)

# Example usage with your `rules` dictionary:
summarize_rule_violations(rules)

,rule_name,affected_rows
0,Invalid/Outlier Amounts,4
1,Missing ID or Amount,2
3,Inconsistent Region Labels,2
2,Duplicate Transactions,1
